In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

In [ ]:
from package_files.benefits_defns import *

In [ ]:
path_all = '../data/us_10m_nointernship_ai_skills_benefits.parquet.gzip'

In [ ]:
usdf = pd.read_parquet(path_all)

In [ ]:
duration_dfs = []
for benefit in benefits4:
    df_duration = usdf.groupby(['AI ROLE',benefit]).agg({'DURATION_CALC':'median'}).reset_index()
    display(df_duration)
    duration_dfs.append(df_duration)

In [ ]:
duration_dfs = []
for benefit in benefits4:
    df_duration = usdf.groupby(['AI ROLE',benefit]).agg({'DURATION_CALC':'mean'}).reset_index()
    display(df_duration)
    duration_dfs.append(df_duration)

In [ ]:
df_duration

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Adjust the figure size for readability of the plots
plt.figure(figsize=(12, 6 * len(benefits4)))

for i, benefit in enumerate(benefits4, 1):
    plt.subplot(len(benefits4), 1, i)  # Set a subplot for each benefit
    sns.boxplot(x='AI ROLE', y='DURATION_CALC', hue=benefit, data=usdf)
    plt.title(f'Duration by AI Role and {benefit}')
    plt.xlabel('AI Role')
    plt.ylabel('Duration')
    plt.legend(title=benefit)
    plt.xticks(rotation=45)  # Rotate x-axis labels for readability

plt.tight_layout()
plt.show()


In [ ]:
plot_data = usdf.melt(id_vars=['AI ROLE', 'DURATION'], value_vars=benefits4, var_name='Benefit', value_name='Has_Benefit')

In [ ]:

# Create a single box plot grouped by AI ROLE and each benefit
plt.figure(figsize=(14, 8))
sns.boxplot(x='AI ROLE', y='DURATION', hue='Benefit', data=usdf.melt(id_vars=['AI ROLE', 'DURATION'], value_vars=benefits4, var_name='Benefit', value_name='Has_Benefit'))
plt.title('Duration by AI Role and Benefits')
plt.xlabel('AI Role')
plt.ylabel('Duration')
plt.legend(title='Benefit', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)  # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()


In [ ]:
df_melted = usdf.melt(id_vars=['AI ROLE', 'DURATION_CALC'], value_vars=benefits4, 
                      var_name='Benefit', value_name='Has_Benefit')

In [ ]:
df_melted

In [ ]:
plt.figure(figsize=(12, 6))

sns.pointplot(x='Benefit', y='DURATION_CALC', hue='AI ROLE', style='Has_Benefit', 
              markers=["o", "^"], data=df_melted, dodge=True, errorbar='sd')

plt.title('Duration by AI Role and Benefits')
plt.ylabel('Duration')
plt.xlabel('Benefit')
plt.xticks(rotation=45)  # Rotate x-axis labels for readability
plt.legend(title='AI Role / Benefit Presence')
plt.tight_layout()
plt.show()

In [ ]:
df_melted['Has_Benefit'] = df_melted['Has_Benefit'].astype(str)

# Use catplot with kind='point' to mimic the style from your image
g = sns.catplot(x='Benefit', y='DURATION_CALC', hue='AI ROLE', col='Has_Benefit', 
                kind='point', data=df_melted, dodge=True, ci='sd', 
                markers=["o", "^"], height=5, aspect=1.5)

# Set plot title and adjust labels
g.fig.suptitle('Duration by AI Role and Benefits', y=1.02)
g.set_axis_labels('Benefit', 'Duration')
g.set_xticklabels(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:

# Reshape the dataframe to long format for easier plotting
df_melted = usdf.melt(id_vars=['AI ROLE', 'DURATION_CALC'], value_vars=benefits4, 
                      var_name='Benefit', value_name='Has_Benefit')

# Convert Has_Benefit to a string for clear category separation
df_melted['Has_Benefit'] = df_melted['Has_Benefit'].astype(str)
df_melted['AI ROLE'] = df_melted['AI ROLE'].astype(str)

# Create a new column combining AI Role and Has Benefit for better grouping in the plot
df_melted['Role_Benefit'] = df_melted['AI ROLE'] + ' / ' + df_melted['Has_Benefit']




In [ ]:
df_melted

In [ ]:
# Plotting
plt.figure(figsize=(14, 8))
sns.boxplot(x='Benefit', y='DURATION_CALC', hue='Role_Benefit', data=df_melted)

plt.title('Duration by AI Role and Benefits')
plt.ylabel('Duration')
plt.xlabel('Benefit')
plt.xticks(rotation=45)  # Rotate x-axis labels for readability
plt.legend(title='AI Role / Benefit Presence', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
benefit_dfs = []
for benefit in benefits4:
    # Create a subset of the data for this specific benefit
    df_subset = usdf[['AI ROLE', 'DURATION_CALC', benefit]].copy()
    df_subset.dropna(subset=[benefit], inplace=True)
    # replace 0s and 1s with False and True in the benefit column
    df_subset[benefit] = df_subset[benefit].replace({0: 'False', 1: 'True'})

    # Rename the benefit column to 'Has_Benefit' for consistency
    df_subset.rename(columns={benefit: 'Has_Benefit'}, inplace=True)

    # Convert Has_Benefit to string for proper category handling
    df_subset['Has_Benefit'] = df_subset['Has_Benefit'].astype(str)
    df_subset['AI ROLE'] = df_subset['AI ROLE'].astype(str)

    # Create a new column combining AI Role and Has Benefit for better grouping in the plot
    df_subset['Role_Benefit'] = df_subset['AI ROLE'] + ' / ' + df_subset['Has_Benefit']
    df_subset['Role_Benefit'] = df_subset['Role_Benefit'].replace({
    'True / True': 'AI Role / Offers Benefit',
    'True / False': 'AI Role / No Benefit',
    'False / True': 'Non-AI Role / Offers Benefit',
    'False / False': 'Non-AI Role / No Benefit'
})
    benefit_dfs.append(df_subset)

In [ ]:
palette = {
    'AI Role / Offers Benefit': '#1f77b4',   # AI Role + Benefit
    'AI Role / No Benefit': '#aec7e8', # AI Role + No Benefit
    'Non-AI Role / Offers Benefit': '#ff7f0e',# Non-AI Role + Benefit
    'Non-AI Role / No Benefit': '#ffbb78' # Non-AI Role + No Benefit
}
# Create a figure for plotting
plt.figure(figsize=(14, 8))
hue_order = ['AI Role / Offers Benefit', 'AI Role / No Benefit', 'Non-AI Role / Offers Benefit', 'Non-AI Role / No Benefit']

for benefit, df_subset in zip(benefits4, benefit_dfs):
    # Plot the boxplot for the current benefit
    sns.boxplot(x=[benefit] * len(df_subset), y='DURATION_CALC', hue='Role_Benefit', data=df_subset, dodge=True, palette=palette)

# Set plot labels and title
plt.title('Duration by AI Role and Benefits')
plt.ylabel('Duration')
plt.xlabel('Benefit')
plt.xticks(ticks=np.arange(len(benefits4))+0.5, labels = benefits4_labels, rotation = 45, ha = 'right')  # Rotate x-axis labels for readability
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles=handles[:4], labels=labels[:4], title='AI Role / Benefit Presence', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Adjust your color palette for the updated Role_Benefit labels
palette = {
    'AI Role / Offers Benefit': '#1f77b4',   # AI Role + Benefit
    'AI Role / No Benefit': '#aec7e8',       # AI Role + No Benefit
    'Non-AI Role / Offers Benefit': '#ff7f0e', # Non-AI Role + Benefit
    'Non-AI Role / No Benefit': '#ffbb78'    # Non-AI Role + No Benefit
}

# Assuming benefit_dfs is a list of separate DataFrames, first combine them into a single DataFrame
# Add a new column 'Benefit' to indicate each benefit in the merged DataFrame
combined_df = pd.concat(
    [df.assign(Benefit=benefit) for benefit, df in zip(benefits4, benefit_dfs)],
    ignore_index=True
)

# Map the Role_Benefit column to the more descriptive labels in the combined DataFrame
combined_df['Role_Benefit'] = combined_df['Role_Benefit'].replace({
    'True / True': 'AI Role / Offers Benefit',
    'True / False': 'AI Role / No Benefit',
    'False / True': 'Non-AI Role / Offers Benefit',
    'False / False': 'Non-AI Role / No Benefit'
})

# Create a figure for plotting
plt.figure(figsize=(14, 8))
hue_order = ['AI Role / Offers Benefit', 'AI Role / No Benefit', 'Non-AI Role / Offers Benefit', 'Non-AI Role / No Benefit']

# Plot all benefits in one go
sns.boxplot(x='Benefit', y='DURATION_CALC', hue='Role_Benefit', data=combined_df, dodge=True, palette=palette, hue_order = hue_order)

# Set plot labels and title
plt.title('Duration by AI Role and Benefits')
plt.ylabel('Duration')
plt.xlabel('Benefit')

# Adjust x-tick labels for readability, rotating as needed
plt.xticks(ticks=np.arange(len(benefits4)), labels=benefits4_labels, rotation=45, ha='right')

# Limit the legend to the four desired labels
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles=handles[:4], labels=labels[:4], title='AI Role / Benefit Presence', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
palette = {
    'AI Role / Offers Benefit': '#1f77b4',   # AI Role + Benefit
    'AI Role / No Benefit': '#aec7e8', # AI Role + No Benefit
    'Non-AI Role / Offers Benefit': '#ff7f0e',# Non-AI Role + Benefit
    'Non-AI Role / No Benefit': '#ffbb78' # Non-AI Role + No Benefit
}
# Create a figure for plotting
plt.figure(figsize=(14, 8))

# Loop through each benefit and create a plot
for benefit in benefits4:
    # Create a subset of the data for this specific benefit
    df_subset = usdf[['AI ROLE', 'DURATION_CALC', benefit]].copy()
    df_subset.dropna(subset=[benefit], inplace=True)
    # replace 0s and 1s with False and True in the benefit column
    df_subset[benefit] = df_subset[benefit].replace({0: 'False', 1: 'True'})

    # Rename the benefit column to 'Has_Benefit' for consistency
    df_subset.rename(columns={benefit: 'Has_Benefit'}, inplace=True)

    # Convert Has_Benefit to string for proper category handling
    df_subset['Has_Benefit'] = df_subset['Has_Benefit'].astype(str)
    df_subset['AI ROLE'] = df_subset['AI ROLE'].astype(str)

    # Create a new column combining AI Role and Has Benefit for better grouping in the plot
    df_subset['Role_Benefit'] = df_subset['AI ROLE'] + ' / ' + df_subset['Has_Benefit']
    df_subset['Role_Benefit'] = df_subset['Role_Benefit'].replace({
    'True / True': 'AI Role / Offers Benefit',
    'True / False': 'AI Role / No Benefit',
    'False / True': 'Non-AI Role / Offers Benefit',
    'False / False': 'Non-AI Role / No Benefit'
})
    
    # Plot the boxplot for the current benefit
    sns.boxplot(x=[benefit] * len(df_subset), y='DURATION_CALC', hue='Role_Benefit', data=df_subset, dodge=True, palette=palette)

# Set plot labels and title
plt.title('Duration by AI Role and Benefits')
plt.ylabel('Duration')
plt.xlabel('Benefit')
plt.xticks(ticks = )  # Rotate x-axis labels for readability
plt.legend(title='AI Role / Benefit Presence', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
df_subset

In [ ]:
benefit_dfs = []
for benefit in benefits4:
    # Create a subset of the data for this specific benefit
    df_subset = usdf[['AI ROLE', 'DURATION_CALC', benefit]].copy()
    df_subset.dropna(subset=[benefit], inplace=True)
    # replace 0s and 1s with False and True in the benefit column
    df_subset[benefit] = df_subset[benefit].replace({0: 'False', 1: 'True'})

    # Rename the benefit column to 'Has_Benefit' for consistency
    df_subset.rename(columns={benefit: 'Has_Benefit'}, inplace=True)

    # Convert Has_Benefit to string for proper category handling
    df_subset['Has_Benefit'] = df_subset['Has_Benefit'].astype(str)
    df_subset['AI ROLE'] = df_subset['AI ROLE'].astype(str)

    # Create a new column combining AI Role and Has Benefit for better grouping in the plot
    df_subset['Role_Benefit'] = df_subset['AI ROLE'] + ' / ' + df_subset['Has_Benefit']
    df_subset['Role_Benefit'] = df_subset['Role_Benefit'].replace({
    'True / True': 'AI Role / Offers Benefit',
    'True / False': 'AI Role / No Benefit',
    'False / True': 'Non-AI Role / Offers Benefit',
    'False / False': 'Non-AI Role / No Benefit'
})
    
    benefit_dfs.append(df_subset)


In [ ]:
df_subset

In [ ]:
palette = {
    'AI Role / Offers Benefit': '#1f77b4',   # AI Role + Benefit
    'AI Role / No Benefit': '#aec7e8', # AI Role + No Benefit
    'Non-AI Role / Offers Benefit': '#ff7f0e',# Non-AI Role + Benefit
    'Non-AI Role / No Benefit': '#ffbb78' # Non-AI Role + No Benefit
}
# Create a figure for plotting
plt.figure(figsize=(14, 8))

# Loop through each benefit and create a plot
for df_subset in benefit_dfs:
    # Plot the boxplot for the current benefit
    print("loop")
    sns.boxplot(x=[benefit] * len(df_subset), y='DURATION_CALC', hue='Role_Benefit', data=df_subset, dodge=True, palette=palette)
# Set plot labels and title
print("plt")
plt.title('Duration by AI Role and Benefits')
plt.ylabel('Duration')
plt.xlabel('Benefit')
plt.xticks(rotation=45)  # Rotate x-axis labels for readability

# Adjust the legend to only show 4 categories
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles=handles[:4], labels=labels[:4], title='AI Role / Benefit Presence', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()